# fase_1 - script_cimut Migration

This notebook handles migration of database from old DB to new DB for fase 1.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [16]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [17]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

# Connect ke DB Future
db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration
Connected to future database: 2


## 2. Ambil Data dari DB Lama

In [18]:
# ---------------------------------------------------------
# UPDATE: AMBIL DAFTAR TABEL SECARA DINAMIS
# ---------------------------------------------------------
cursor_old.execute("SHOW TABLES")
tables_data = cursor_old.fetchall()

# Mengambil nama tabel dari hasil query
# Note: Format output 'SHOW TABLES' biasanya {'Tables_in_dbname': 'tablename'}
target_tables = [list(t.values())[0] for t in tables_data]

print(f"\n--- Ditemukan {len(target_tables)} tabel di Database Lama ---")
print(target_tables)

# Dictionary untuk menyimpan data yang sudah di-load
df_old = {}

print("\n--- Memulai proses load semua data tabel ---")

for table in target_tables:
    try:
        # Load data menggunakan pandas langsung dari koneksi SQL
        query = f"SELECT * FROM `{table}`"
        df_old[table] = pd.read_sql(query, db_old)
        
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_old[table])}")
        
    except Exception as e:
        print(f"Gagal load tabel {table}: {e}")

print("\n--- Proses load selesai. Semua data tersimpan di 'df_old' ---")
print("Kamu sekarang bisa akses datanya dengan: df_old['nama_tabel']")

# Contoh akses data:
# print(df_old['users'].head())

# Tutup koneksi jika sudah tidak digunakan
# db_old.close()
# db_new.close()


--- Ditemukan 108 tabel di Database Lama ---
['absensi', 'absensi_note', 'bidang', 'bidangkategori', 'bidanglink', 'calon', 'calon_detil', 'calon_pertanyaan', 'calon_pertanyaan_detil', 'catatan_kelas', 'catatan_kelas_tag', 'catatan_mingguan', 'catatan_siswa', 'catatan_siswa_follow_up', 'catatanawal_admin', 'catatanawal_datautama', 'catatanawal_infolain', 'catatanawal_tglpenting', 'divisi', 'docs', 'file_rapor_siswa', 'form', 'form_calon', 'form_calon_detil1', 'form_calon_detil2', 'form_calon_detil3', 'form_calon_detil4', 'format_rapor', 'format_rapor_detil', 'format_rapor_detil_rumus', 'format_rapor_rumus', 'format_raport_level', 'hakakses', 'histori_pengajuan', 'history_rapor', 'identitas', 'infrastruktur', 'jabatan', 'jadwal', 'jadwal_detil', 'jadwal_pengajar', 'jadwal_siswa', 'jamkerja', 'kabupaten', 'karyawan', 'kecamatan', 'keluar', 'keluarga', 'kelurahan', 'kurikulum', 'kurikulum_detil', 'kurikulum_detil_sub', 'kurikulum_kelas', 'kursus', 'leapprofil', 'leapverse', 'level', 'lib

In [19]:
# ---------------------------------------------------------
# UPDATE: AMBIL DAFTAR TABEL SECARA DINAMIS (DATABASE BARU)
# ---------------------------------------------------------
# Menggunakan cursor dari database baru
cursor_new.execute("SHOW TABLES")
tables_data_new = cursor_new.fetchall()

# Mengambil nama tabel dari hasil query
target_tables_new = [list(t.values())[0] for t in tables_data_new]

print(f"\n--- Ditemukan {len(target_tables_new)} tabel di Database Baru ---")
print(target_tables_new)

# Dictionary untuk menyimpan data dari database baru (jika diperlukan untuk verifikasi)
df_new = {}

print("\n--- Memulai proses load semua data dari Database Baru ---")

for table in target_tables_new:
    try:
        # Load data menggunakan pandas dengan koneksi database baru
        query = f"SELECT * FROM `{table}`"
        df_new[table] = pd.read_sql(query, db_new)
        
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_new[table])}")
        
    except Exception as e:
        print(f"Gagal load tabel {table} dari DB Baru: {e}")

print("\n--- Proses load selesai. Data DB Baru tersimpan di 'df_new' ---")


--- Ditemukan 104 tabel di Database Baru ---
['absensi', 'activity_log', 'admin_sarpras', 'bidang_kategori', 'bidang_link', 'busdev_bidang', 'cache', 'cache_locks', 'calon_siswa', 'calon_siswa_akademik', 'calon_siswa_bayar', 'calon_siswa_jadwal', 'calon_siswa_kursus', 'calon_siswa_ortu', 'calon_siswa_proses', 'calon_siswa_status_logs', 'catatan_kelas', 'catatan_kelas_tag', 'catatan_mingguan', 'catatan_siswa', 'division_user', 'divisions', 'failed_jobs', 'followup_cs', 'histori_pengajuan', 'izin_karyawan', 'jadwal', 'jadwal_detail', 'jadwal_detail_logs', 'jadwal_hari', 'jadwal_pengajar', 'jadwal_siswa', 'job_batches', 'jobs', 'kabupaten', 'karyawan', 'karyawan_resign', 'kecamatan', 'keluarga_karyawan', 'kelurahan', 'kemitraan_verifikator', 'kontak_prospek', 'kursus', 'kursus_level', 'kursus_libur', 'kursus_siswa', 'level', 'libur', 'log_aktivitas', 'migrations', 'mitra', 'mitra_progres', 'model_has_permissions', 'model_has_roles', 'mou', 'parameter_nilai', 'password_reset_tokens', 'pel

In [20]:
# ---------------------------------------------------------
# UPDATE: AMBIL DAFTAR TABEL SECARA DINAMIS (DATABASE BARU)
# ---------------------------------------------------------
# Menggunakan cursor dari database baru
cursor_future.execute("SHOW TABLES")
tables_data_future = cursor_future.fetchall()

# Mengambil nama tabel dari hasil query
target_tables_future = [list(t.values())[0] for t in tables_data_future]

print(f"\n--- Ditemukan {len(target_tables_future)} tabel di Database Future ---")
print(target_tables_future)

# Dictionary untuk menyimpan data dari database future (jika diperlukan untuk verifikasi)
df_future = {}

print("\n--- Memulai proses load semua data dari Database Future ---")

for table in target_tables_future:
    try:
        # Load data menggunakan pandas dengan koneksi database future
        query = f"SELECT * FROM `{table}`"
        df_future[table] = pd.read_sql(query, db_future)
        
        print(f"Berhasil load tabel: {table} | Jumlah baris: {len(df_future[table])}")
        
    except Exception as e:
        print(f"Gagal load tabel {table} dari DB Future: {e}")

print("\n--- Proses load selesai. Data DB Future tersimpan di 'df_future' ---")


--- Ditemukan 113 tabel di Database Future ---
['absensi', 'activity_log', 'admin_sarpras', 'bidang_kategori', 'bidang_link', 'busdev_bidang', 'cache', 'cache_locks', 'calon_siswa', 'calon_siswa_akademik', 'calon_siswa_bayar', 'calon_siswa_fo_detail', 'calon_siswa_form_program_requirements', 'calon_siswa_form_programs', 'calon_siswa_jadwal', 'calon_siswa_kursus', 'calon_siswa_ortu', 'calon_siswa_proses', 'calon_siswa_proses_logs', 'calon_siswa_status_logs', 'catatan_kelas', 'catatan_kelas_tag', 'catatan_mingguan', 'catatan_remidi_siswa', 'catatan_siswa', 'division_user', 'divisions', 'failed_jobs', 'followup_cs', 'histori_pengajuan', 'izin_karyawan', 'jadwal', 'jadwal_detail', 'jadwal_detail_logs', 'jadwal_hari', 'jadwal_pengajar', 'jadwal_siswa', 'job_batches', 'jobs', 'kabupaten', 'karyawan', 'karyawan_resign', 'kecamatan', 'keluarga_karyawan', 'kelurahan', 'kemitraan_verifikator', 'kontak_prospek', 'kursus', 'kursus_level', 'kursus_libur', 'kursus_siswa', 'level', 'libur', 'log_akt

## 3. Transform Data (jika diperlukan)

users, divisions, shift_kerja, admin_sarpras, sop_kategori, provinsi, web_berita, web_statistik.


In [21]:
df_new['shift_kerja']['nama_shift'] = df_old['jamkerja']['namajamkerja']
df_new['shift_kerja']['jam_masuk']  = df_old['jamkerja']['jammasuk']
df_new['shift_kerja']['jam_pulang'] = df_old['jamkerja']['jampulang']

df_new['admin_sarpras']['wa_admin_sarpras'] = df_old['nowag']['wa']

df_new['sop_kategori']['nama_kategori_sop'] = df_old['sopkategori']['nama']

print("--- Info Tabel Baru (shift_kerja) ---")
df_new['shift_kerja'].info()
print("--- Info Tabel Baru (admin_sarpras) ---")
df_new['admin_sarpras'].info()
print("--- Info Tabel Baru (sop_kategori) ---")
df_new['sop_kategori'].info()

--- Info Tabel Baru (shift_kerja) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype          
---  ------      --------------  -----          
 0   id_shift    3 non-null      int64          
 1   nama_shift  3 non-null      object         
 2   jam_masuk   3 non-null      timedelta64[ns]
 3   jam_pulang  3 non-null      timedelta64[ns]
dtypes: int64(1), object(1), timedelta64[ns](2)
memory usage: 228.0+ bytes
--- Info Tabel Baru (admin_sarpras) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_admin_sarpras  1 non-null      int64 
 1   wa_admin_sarpras  1 non-null      object
dtypes: int64(1), object(1)
memory usage: 148.0+ bytes
--- Info Tabel Baru (sop_kategori) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data co

In [22]:
# Pastikan DataFrame sudah terisi
df_users = df_old['users']

print("--- 1. Pengecekan Nilai Kosong (NULL) ---")
# Cek berapa banyak data kosong di tiap kolom
print(df_users.isnull().sum())

print("\n--- 2. Pengecekan Data Duplikat ---")
# Cek jumlah baris duplikat
duplicate_count = df_users.duplicated().sum()
print(f"Jumlah baris duplikat: {duplicate_count}")

if duplicate_count > 0:
    print("\n--- 3. Contoh Data Duplikat ---")
    # Menampilkan data yang duplikat agar kamu bisa evaluasi
    print(df_users[df_users.duplicated(keep=False)].head())

print("\n--- 4. Pengecekan Format Email (Opsional) ---")
# Contoh: Cek apakah ada email yang tidak mengandung '@' (sederhana)
invalid_emails = df_users[~df_users['email'].str.contains('@', na=False)]
if not invalid_emails.empty:
    print(f"Ada {len(invalid_emails)} email yang formatnya aneh:")
    print(invalid_emails[['email']])
else:
    print("Semua format email terlihat aman.")

--- 1. Pengecekan Nilai Kosong (NULL) ---
idusers        0
email          0
pass           0
nama           1
foto           7
idrole         0
wa             1
thnbekerja     0
idjabatan      0
idjamkerja    37
minat          1
status         0
idbidang      41
ispurchase     1
isteaching     1
ishr           0
isga           0
isit           0
ispdd          0
isbusdev       0
ispimpinan     0
ttd           36
expertise     12
dtype: int64

--- 2. Pengecekan Data Duplikat ---
Jumlah baris duplikat: 0

--- 4. Pengecekan Format Email (Opsional) ---
Semua format email terlihat aman.


In [23]:
df_old['users'].loc[df_old['users']['idusers'] == 'U00046', 'nama'] = 'sosialmedia1'

In [24]:
df_old['users'].isnull().sum()

idusers        0
email          0
pass           0
nama           0
foto           7
idrole         0
wa             1
thnbekerja     0
idjabatan      0
idjamkerja    37
minat          1
status         0
idbidang      41
ispurchase     1
isteaching     1
ishr           0
isga           0
isit           0
ispdd          0
isbusdev       0
ispimpinan     0
ttd           36
expertise     12
dtype: int64

In [25]:
import pandas as pd
import datetime

print("================================================================================")
print(" 🚀 FASE 1 - STEP 1: TRANSFORMASI TABEL USERS (WITH TIMESTAMP FIXED) 🚀 ")
print("================================================================================")

# Ambil waktu saat ini sebagai penanda waktu migrasi resmi
current_time = datetime.datetime.now()

# 🎯 Sesuai arahan Cimut, pasangkan kolom dasar db_old ke wadah df_new
df_new['users']['id_user'] = df_old['users']['idusers']
df_new['users']['name'] = df_old['users']['nama']
df_new['users']['email'] = df_old['users']['email']
df_new['users']['password'] = df_old['users']['pass']

# 🆕 PENAMBAHAN BAGIAN UPDATE BARU: Suntik otomatis kolom timestamp sistem Laravel
df_new['users']['email_verified_at'] = current_time
df_new['users']['created_at'] = current_time
df_new['users']['updated_at'] = current_time

# Tampilkan hasil verifikasi struktur akhir
print("\n✓ Sukses menyinkronkan data users dengan timestamp masa depan!")
print("--- Preview data yang akan diinsert ke tabel users ---")
df_new['users'].info()
display(df_new['users'])

 🚀 FASE 1 - STEP 1: TRANSFORMASI TABEL USERS (WITH TIMESTAMP FIXED) 🚀 

✓ Sukses menyinkronkan data users dengan timestamp masa depan!
--- Preview data yang akan diinsert ke tabel users ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_user            51 non-null     object        
 1   name               51 non-null     object        
 2   email              51 non-null     object        
 3   email_verified_at  51 non-null     datetime64[us]
 4   password           51 non-null     object        
 5   remember_token     0 non-null      object        
 6   created_at         51 non-null     datetime64[us]
 7   updated_at         51 non-null     datetime64[us]
dtypes: datetime64[us](3), object(5)
memory usage: 3.3+ KB


,id_user,name,email,email_verified_at,password,remember_token,created_at,updated_at
0,U00001,ADMINISTRATOR,ditari@leapsurabaya.sch.id,2026-06-06 22:57:58.616115,aGtq,None,2026-06-06 22:57:58.616115,2026-06-06 22:57:58.616115
1,U00003,"Graciela Evanda Ronadi, S.Kom.",graciela@leapsurabaya.sch.id,2026-06-06 22:57:58.616115,qWmlbcVjYmo%3D,None,2026-06-06 22:57:58.616115,2026-06-06 22:57:58.616115
2,U00011,DANIAR AULIA RIZKI,daniar.rizki@leapsurabaya.sch.id,2026-06-06 22:57:58.616115,aGtq,None,2026-06-06 22:57:58.616115,2026-06-06 22:57:58.616115
3,U00012,Habibah Melyna,habibah.elfiani@leapsurabaya.sch.id,2026-06-06 22:57:58.616115,o56YqZJkZA%3D%3D,None,2026-06-06 22:57:58.616115,2026-06-06 22:57:58.616115
4,U00014,Laksmi Puspitowardhani,laksmi.p@leapsurabaya.sch.id,2026-06-06 22:57:58.616115,aWlpbw%3D%3D,None,2026-06-06 22:57:58.616115,2026-06-06 22:57:58.616115
5,U00015,Ika Asriani Yadin,ikayadin@leapsurabaya.sch.id,2026-06-06 22:57:58.616115,aGtq,None,2026-06-06 22:57:58.616115,2026-06-06 22:57:58.616115
6,U00016,"Luluk Fatikah Sari, S.Pd.",luluk.sari@leapsurabaya.sch.id,2026-06-06 22:57:58.616115,o66jrsxjY2s%3D,None,2026-06-06 22:57:58.616115,2026-06-06 22:57:58.616115
7,U00018,Ditari Kurnia,admin@gmail.com,2026-06-06 22:57:58.616115,aGtq,None,2026-06-06 22:57:58.616115,2026-06-06 22:57:58.616115
8,U00019,"Hartatik, S.S.",hartatik@leapsurabaya.sch.id,2026-06-06 22:57:58.616115,p5qenpJkZA%3D%3D,None,2026-06-06 22:57:58.616115,2026-06-06 22:57:58.616115
9,U00020,Juni Arlianto,juni.arlianto@leapsurabaya.sch.id,2026-06-06 22:57:58.616115,aGtq,None,2026-06-06 22:57:58.616115,2026-06-06 22:57:58.616115


In [26]:
# Analisis tabel df_old['divisi']

print("=" * 60)
print("ANALISIS TABEL: df_old['divisi']")
print("=" * 60)

# 1. Lihat struktur dan isi data
print("\n--- 1. STRUKTUR DATA ---")
print(f"Shape: {df_old['divisi'].shape}")
print(f"Jumlah baris: {len(df_old['divisi'])}")
print(f"Jumlah kolom: {len(df_old['divisi'].columns)}")

print("\n--- 2. NAMA KOLOM ---")
print(df_old['divisi'].columns.tolist())

print("\n--- 3. PREVIEW DATA ---")
print(df_old['divisi'])

print("\n--- 4. INFO TABEL ---")
df_old['divisi'].info()

print("\n--- 5. DATA TYPES ---")
print(df_old['divisi'].dtypes)

print("\n--- 6. PENGECEKAN NULL/KOSONG ---")
null_counts_divisi = df_old['divisi'].isnull().sum()
print(null_counts_divisi)
if null_counts_divisi.sum() > 0:
    print(f"\n⚠️ Ada {null_counts_divisi.sum()} nilai kosong ditemukan")
else:
    print("\n✓ Tidak ada nilai kosong")

print("\n--- 7. PENGECEKAN DUPLIKAT ---")
duplicate_rows_divisi = df_old['divisi'].duplicated().sum()
print(f"Jumlah baris duplikat: {duplicate_rows_divisi}")
if duplicate_rows_divisi > 0:
    print("\nData duplikat:")
    print(df_old['divisi'][df_old['divisi'].duplicated(keep=False)])

print("\n--- 8. STATISTIK DESKRIPTIF ---")
print(df_old['divisi'].describe(include='all'))

print("\n--- 9. UNIQUE VALUES PER KOLOM ---")
for col_divisi in df_old['divisi'].columns:
    print(f"{col_divisi}: {df_old['divisi'][col_divisi].nunique()} unique values")

print("\n--- 10. VALUE COUNTS (untuk kolom kategori) ---")
for col_divisi in df_old['divisi'].columns:
    print(f"\n{col_divisi}:")
    print(df_old['divisi'][col_divisi].value_counts(dropna=False))

ANALISIS TABEL: df_old['divisi']

--- 1. STRUKTUR DATA ---
Shape: (6, 2)
Jumlah baris: 6
Jumlah kolom: 2

--- 2. NAMA KOLOM ---
['iddivisi', 'nama']

--- 3. PREVIEW DATA ---
  iddivisi        nama
0   D00001          IT
1   D00002      Busdev
2   D00003     HR / GA
3   D00005  Pendidikan
4   D00006     Finance
5   D00007     Direksi

--- 4. INFO TABEL ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   iddivisi  6 non-null      object
 1   nama      6 non-null      object
dtypes: object(2)
memory usage: 228.0+ bytes

--- 5. DATA TYPES ---
iddivisi    object
nama        object
dtype: object

--- 6. PENGECEKAN NULL/KOSONG ---
iddivisi    0
nama        0
dtype: int64

✓ Tidak ada nilai kosong

--- 7. PENGECEKAN DUPLIKAT ---
Jumlah baris duplikat: 0

--- 8. STATISTIK DESKRIPTIF ---
       iddivisi nama
count         6    6
unique        6    6
top      D00001

In [27]:
df_future['roles'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           0 non-null      object
 1   id_division  0 non-null      object
 2   name         0 non-null      object
 3   guard_name   0 non-null      object
 4   created_at   0 non-null      object
 5   updated_at   0 non-null      object
dtypes: object(6)
memory usage: 132.0+ bytes


In [28]:
# Mapping data dari df_old['divisi'] ke df_new['divisions']
df_new['divisions']['id_division'] = df_old['divisi']['iddivisi']
df_new['divisions']['name_division'] = df_old['divisi']['nama']
df_new['divisions']['description'] = None
df_new['divisions']['is_active'] = 1

print("--- Preview data divisions yang sudah dimapping ---")
display(df_new['divisions'])
print("\n--- Info tabel divisions ---")
display(df_new['divisions'].info())

--- Preview data divisions yang sudah dimapping ---


,id_division,name_division,description,is_active
0,D00001,IT,None,1
1,D00002,Busdev,None,1
2,D00003,HR / GA,None,1
3,D00005,Pendidikan,None,1
4,D00006,Finance,None,1
5,D00007,Direksi,None,1
6,NaN,NaN,None,1
7,NaN,NaN,None,1
8,NaN,NaN,None,1
9,NaN,NaN,None,1



--- Info tabel divisions ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id_division    6 non-null      object
 1   name_division  6 non-null      object
 2   description    0 non-null      object
 3   is_active      12 non-null     int64 
dtypes: int64(1), object(3)
memory usage: 516.0+ bytes


None

In [29]:
df_old['role_users'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   idru     0 non-null      object
 1   idusers  0 non-null      object
 2   idmitra  0 non-null      object
dtypes: object(3)
memory usage: 132.0+ bytes


In [31]:
print("="*80)
print("RINGKASAN DATA YANG SUDAH DIPROSES")
print("="*80)

tables_to_show = ['users', 'divisions', 'shift_kerja', 'admin_sarpras', 'sop_kategori']

for table_name in tables_to_show:
    print(f"\n{'='*80}")
    print(f"TABEL: {table_name}")
    print(f"{'='*80}")
    
    if table_name == 'users':
        source_df = df_new['users']
        print(f"\nJumlah baris: {len(source_df)}")
        print(f"Jumlah kolom: {len(source_df.columns)}\n")
        print(source_df.to_string())
        
    elif table_name == 'divisions':
        source_df = df_new['divisions']
        print(f"\nJumlah baris: {len(source_df)}")
        print(f"Jumlah kolom: {len(source_df.columns)}\n")
        print(source_df.to_string())
        
    elif table_name == 'shift_kerja':
        source_df = df_new['shift_kerja']
        print(f"\nJumlah baris: {len(source_df)}")
        print(f"Jumlah kolom: {len(source_df.columns)}\n")
        print(source_df.to_string())
        
    elif table_name == 'admin_sarpras':
        source_df = df_new['admin_sarpras']
        print(f"\nJumlah baris: {len(source_df)}")
        print(f"Jumlah kolom: {len(source_df.columns)}\n")
        print(source_df.to_string())
        
    elif table_name == 'sop_kategori':
        source_df = df_new['sop_kategori']
        print(f"\nJumlah baris: {len(source_df)}")
        print(f"Jumlah kolom: {len(source_df.columns)}\n")
        print(source_df.to_string())

print(f"\n{'='*80}")
print("SELESAI")
print(f"{'='*80}")

RINGKASAN DATA YANG SUDAH DIPROSES

TABEL: users

Jumlah baris: 51
Jumlah kolom: 8

   id_user                                    name                                    email          email_verified_at                password remember_token                 created_at                 updated_at
0   U00001                           ADMINISTRATOR               ditari@leapsurabaya.sch.id 2026-06-06 22:57:58.616115                    aGtq           None 2026-06-06 22:57:58.616115 2026-06-06 22:57:58.616115
1   U00003          Graciela Evanda Ronadi, S.Kom.             graciela@leapsurabaya.sch.id 2026-06-06 22:57:58.616115          qWmlbcVjYmo%3D           None 2026-06-06 22:57:58.616115 2026-06-06 22:57:58.616115
2   U00011                      DANIAR AULIA RIZKI         daniar.rizki@leapsurabaya.sch.id 2026-06-06 22:57:58.616115                    aGtq           None 2026-06-06 22:57:58.616115 2026-06-06 22:57:58.616115
3   U00012                          Habibah Melyna      habibah.elfi

In [32]:
import json
import pickle
with open('fase_1_cimut.pkl', 'wb') as f:
    pickle.dump(df_new, f)

print("✓ Data df_new sudah disimpan ke df_new.pkl")
print("Siap untuk digunakan di insert_handler.ipynb")

✓ Data df_new sudah disimpan ke df_new.pkl
Siap untuk digunakan di insert_handler.ipynb
